In [4]:
import json
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load classes
with open("classes.json", "r") as f:
    class_names = json.load(f)

# Recreate architecture
def load_medicinal_plant_model(model_weights_path):
    model = models.efficientnet_v2_s(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=0.4),
        nn.Linear(512, len(class_names))
    )
    model.load_state_dict(torch.load(model_weights_path, map_location=device))
    model.to(device)
    model.eval()
    return model

inference_model = load_medicinal_plant_model("medicinal_plant_model.pth")

inference_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_plant(image_path, top_k=3):
    img = Image.open(image_path).convert("RGB")
    tensor = inference_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = inference_model(tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    top_probs, top_indices = torch.topk(probabilities, top_k)
    results = [
        {"plant": class_names[idx.item()], "confidence": float(prob.item() * 100)}
        for prob, idx in zip(top_probs, top_indices)
    ]
    return results

# Example call:
print(predict_plant(r"j.16.jpg"))

C:\Users\harsh\AppData\Local\Temp\ipykernel_2240\3024273207.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_weights_path, map_loc